# Data Retrieval

#### Objective
This notebook retrieves the historical stock market data. We are using the **Yahoo Finance API** to pull data for three specific ETFs:
* **SPY** (S&P 500 ETF) - Top 500 US companies
* **QQQ** (NASDAQ ETF) - Top 100 NASDAQ (tech-heavy) companies
* **VTI** (Total Market ETF) - The entire US stock market

We are extracting the maximum available historical data. The raw files are saved to `data/raw/` with no modifications

In [1]:
import yfinance as yf
import pandas as pd
import os

RAW_DIR = "../data/raw" # Define the path to our raw data folder (relative to this notebook)

etfs = ["SPY", "QQQ", "VTI"]
RAW_DIR    = "../data/raw"    

os.makedirs(RAW_DIR, exist_ok=True) # Ensure the raw directory exists

## Download and Save Raw ETF Data

### What gets saved:
One CSV per ETF in `data/raw/` 
— `SPY_raw.csv`, `QQQ_raw.csv`, `VTI_raw.csv`  

In [2]:
for ticker in etfs:
    print(f"Downloading {ticker}.")
    
    # Fetch data from Yahoo Finance API
    df = yf.download(
        ticker,
        period="max", # Get all available historical data
        auto_adjust=False, # Prevent library from dropping the 'Adj Close' column
        multi_level_index=False,  # Ensures a flat DataFrame for easy column renaming
    )
    
    df.reset_index(inplace=True)
    
    # Rename 'Adj Close' to 'Adjusted Close'
    if 'Adj Close' in df.columns:
        df.rename(columns={'Adj Close': 'Adjusted Close'}, inplace=True)
    
    df["Ticker"] = ticker # Add a 'Ticker' column to identify the data 
    
    # Define the exact column orderfor final CSV output
    column_order = ['Date', 'Ticker', 'Open', 'High', 'Low', 'Close', 'Adjusted Close', 'Volume']
    df = df[column_order]
    
    # Save to the raw directory as CSV files
    output_path = os.path.join(RAW_DIR, f"{ticker}_raw.csv")
    df.to_csv(output_path, index=False)
    
    print(f"Saved {len(df)} rows → {output_path}") # Print a success message with the no. of rows and the file path
    print(f"Date range: {df['Date'].min()} to {df['Date'].max()}\n") # Print a success message with the date range captured

[*********************100%***********************]  1 of 1 completed


Saved 8332 rows → ../data/raw\SPY_raw.csv
Date range: 1993-01-29 00:00:00 to 2026-03-06 00:00:00



[*********************100%***********************]  1 of 1 completed


Saved 6790 rows → ../data/raw\QQQ_raw.csv
Date range: 1999-03-10 00:00:00 to 2026-03-06 00:00:00



[*********************100%***********************]  1 of 1 completed

Saved 6217 rows → ../data/raw\VTI_raw.csv
Date range: 2001-06-15 00:00:00 to 2026-03-06 00:00:00



## Sanity Check & Validation
We verify that our raw fileshave been created successfully and contain the correct columns.

In [3]:
for ticker in etfs:
    path = os.path.join(RAW_DIR, f"{ticker}_raw.csv")
    df   = pd.read_csv(path)

    print(f"{ticker}")
    print(f"Rows: {len(df)}")
    print(f"Columns: {list(df.columns)}")
    print(f"Nulls:\n{df.isnull().sum()}") # Check for any missing/null values in the raw data
    
    print("\nPreview:")
    print(df.head(10))

SPY
Rows: 8332
Columns: ['Date', 'Ticker', 'Open', 'High', 'Low', 'Close', 'Adjusted Close', 'Volume']
Nulls:
Date              0
Ticker            0
Open              0
High              0
Low               0
Close             0
Adjusted Close    0
Volume            0
dtype: int64

Preview:
         Date Ticker      Open      High       Low     Close  Adjusted Close  \
0  1993-01-29    SPY  43.96875  43.96875  43.75000  43.93750       24.241413   
1  1993-02-01    SPY  43.96875  44.25000  43.96875  44.25000       24.413820   
2  1993-02-02    SPY  44.21875  44.37500  44.12500  44.34375       24.465532   
3  1993-02-03    SPY  44.40625  44.84375  44.37500  44.81250       24.724167   
4  1993-02-04    SPY  44.96875  45.09375  44.46875  45.00000       24.827614   
5  1993-02-05    SPY  44.96875  45.06250  44.71875  44.96875       24.810377   
6  1993-02-08    SPY  44.96875  45.12500  44.90625  44.96875       24.810377   
7  1993-02-09    SPY  44.81250  44.81250  44.56250  44.65625       